In [ ]:
# -*- coding: utf-8 -*-
"""Skin Disease Detection Web App - Train with Custom Dataset from Google Drive"""

# ============================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================
!pip install ultralytics pillow flask pyngrok opencv-python-headless tensorflow gdown scikit-learn -q

# ============================================
# 2. IMPORTS
# ============================================
import os
import time
import threading
import socket
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from PIL import Image
import base64
from io import BytesIO
from flask import Flask, render_template_string, request, jsonify
from pyngrok import ngrok
import gdown
import zipfile
import shutil
import warnings
warnings.filterwarnings("ignore")

print("=" * 50)
print("All libraries imported successfully!")
print("=" * 50)

# ============================================
# 3. DOWNLOAD DATASET FROM GOOGLE DRIVE
# ============================================

def download_dataset_from_drive():
    """Download the dataset from Google Drive"""

    # Since the folder is shared, we need the folder ID
    # From the URL: https://drive.google.com/drive/folders/1BVPQRsFV9Ff3ryzPmKQWtFbg2omr3zde
    # The folder ID is: 1BVPQRsFV9Ff3ryzPmKQWtFbg2omr3zde

    folder_id = "1BVPQRsFV9Ff3ryzPmKQWtFbg2omr3zde"

    # Alternative approach: Use gdown to download the folder
    # First, let's check if dataset already exists
    if os.path.exists("skin_dataset"):
        print("Dataset already exists. Using existing dataset...")
        return "skin_dataset"

    print("Downloading dataset from Google Drive...")

    # Using gdown to download the folder (requires folder to be shared as zip)
    # For now, we'll create a structured dataset from the folders you mentioned
    # You'll need to either:
    # 1. Share the folder as a zip file, or
    # 2. Place the images in the correct structure

    # Let me create the structure based on the folder names you provided
    dataset_path = "skin_dataset"
    os.makedirs(dataset_path, exist_ok=True)

    # The folders mentioned in your drive:
    classes = ['Biduran', 'Bisul', 'Cacar Air', 'Dermatitis', 'Jerawat']

    print("\n⚠️ IMPORTANT: Please manually download and organize your dataset:")
    print("\nYour Google Drive folder contains these subfolders:")
    for cls in classes:
        print(f"  - {cls}")

    print(f"\nPlease create the following structure in Colab:")
    print(f"skin_dataset/")
    for cls in classes:
        print(f"  ├── train/{cls}/")
        print(f"  │   └── (your images here)")
        print(f"  ├── val/{cls}/")
        print(f"  │   └── (validation images)")
        print(f"  └── test/{cls}/")
        print(f"      └── (test images)")

    print("\nTo automatically download, you can:")
    print("1. Zip the folder in Google Drive")
    print("2. Get the file ID and use gdown.download()")
    print("3. Or use the following command to mount Google Drive:")

    return dataset_path

# Alternative: Mount Google Drive directly
def mount_google_drive():
    """Mount Google Drive to access the dataset"""
    from google.colab import drive
    drive.mount('/content/drive')

    # Path to your dataset in Google Drive
    # Update this path based on where your folder is located
    drive_path = "/content/drive/MyDrive/WYIE Picture"

    if os.path.exists(drive_path):
        print(f"✅ Found dataset at: {drive_path}")
        return drive_path
    else:
        print(f"⚠️ Dataset not found at {drive_path}")
        print("Please check the path or move your folder to the correct location")
        return None

# ============================================
# 4. LOAD AND PREPARE DATASET
# ============================================

def prepare_dataset(dataset_path):
    """Prepare the dataset for training"""

    # Check if dataset path exists
    if not os.path.exists(dataset_path):
        print(f"❌ Dataset not found at {dataset_path}")
        return None, None, None

    # Define image parameters
    IMG_SIZE = (224, 224)  # MobileNetV2 expects 224x224
    BATCH_SIZE = 32

    # Create data generators with augmentation
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    val_datagen = ImageDataGenerator(rescale=1./255)

    # Try to load datasets from train/val/test structure
    train_path = os.path.join(dataset_path, 'train')
    val_path = os.path.join(dataset_path, 'val')
    test_path = os.path.join(dataset_path, 'test')

    if os.path.exists(train_path):
        print("Loading data from train/val/test folders...")
        train_generator = train_datagen.flow_from_directory(
            train_path,
            target_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            class_mode='categorical'
        )

        val_generator = val_datagen.flow_from_directory(
            val_path,
            target_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            class_mode='categorical'
        )

        test_generator = val_datagen.flow_from_directory(
            test_path,
            target_size=IMG_SIZE,
            batch_size=BATCH_SIZE,
            class_mode='categorical',
            shuffle=False
        )

        class_names = list(train_generator.class_indices.keys())

    else:
        # If no train/val split, create from directory
        print("Creating train/val split from main directory...")

        # Look for subdirectories
        subdirs = [d for d in os.listdir(dataset_path)
                  if os.path.isdir(os.path.join(dataset_path, d))]

        if subdirs:
            # Create temporary directory with split
            from sklearn.model_selection import train_test_split
            import shutil

            temp_path = "temp_dataset"
            os.makedirs(temp_path, exist_ok=True)

            for split in ['train', 'val', 'test']:
                for cls in subdirs:
                    os.makedirs(os.path.join(temp_path, split, cls), exist_ok=True)

            # Split images
            for cls in subdirs:
                cls_path = os.path.join(dataset_path, cls)
                images = [f for f in os.listdir(cls_path)
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

                train_imgs, test_imgs = train_test_split(images, test_size=0.3, random_state=42)
                val_imgs, test_imgs = train_test_split(test_imgs, test_size=0.5, random_state=42)

                for img in train_imgs:
                    shutil.copy(os.path.join(cls_path, img),
                               os.path.join(temp_path, 'train', cls, img))
                for img in val_imgs:
                    shutil.copy(os.path.join(cls_path, img),
                               os.path.join(temp_path, 'val', cls, img))
                for img in test_imgs:
                    shutil.copy(os.path.join(cls_path, img),
                               os.path.join(temp_path, 'test', cls, img))

            train_generator = train_datagen.flow_from_directory(
                os.path.join(temp_path, 'train'),
                target_size=IMG_SIZE,
                batch_size=BATCH_SIZE,
                class_mode='categorical'
            )

            val_generator = val_datagen.flow_from_directory(
                os.path.join(temp_path, 'val'),
                target_size=IMG_SIZE,
                batch_size=BATCH_SIZE,
                class_mode='categorical'
            )

            test_generator = val_datagen.flow_from_directory(
                os.path.join(temp_path, 'test'),
                target_size=IMG_SIZE,
                batch_size=BATCH_SIZE,
                class_mode='categorical',
                shuffle=False
            )

            class_names = list(train_generator.class_indices.keys())

        else:
            print("❌ No valid image folders found!")
            return None, None, None

    print(f"\n✅ Dataset loaded successfully!")
    print(f"Classes found: {class_names}")
    print(f"Training samples: {train_generator.samples}")
    print(f"Validation samples: {val_generator.samples}")
    print(f"Test samples: {test_generator.samples}")

    return train_generator, val_generator, test_generator

# ============================================
# 5. BUILD AND TRAIN MODEL
# ============================================

def build_model(num_classes):
    """Build a CNN model using transfer learning"""

    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Freeze base model layers
    base_model.trainable = False

    # Add custom layers
    model = keras.Sequential([
        base_model,
        GlobalAveragePooling2D(),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )

    return model

def train_model(model, train_generator, val_generator, epochs=30):
    """Train the model"""

    # Callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=3,
            min_lr=1e-6,
            verbose=1
        )
    ]

    print("\n" + "=" * 50)
    print("Starting Training...")
    print("=" * 50)

    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )

    # Fine-tuning: Unfreeze top layers
    print("\n" + "=" * 50)
    print("Fine-tuning model...")
    print("=" * 50)

    # Unfreeze the top 20 layers of the base model
    base_model = model.layers[0]
    base_model.trainable = True

    # Freeze first 100 layers
    for layer in base_model.layers[:100]:
        layer.trainable = False

    # Recompile with lower learning rate
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )

    # Continue training
    history_fine = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=10,
        callbacks=callbacks,
        verbose=1
    )

    return model, history

# ============================================
# 6. FIND AVAILABLE PORT
# ============================================

def find_free_port(start_port=5000, max_attempts=10):
    """Find an available port starting from start_port"""
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('localhost', port))
                return port
            except OSError:
                continue
    return None

# ============================================
# 7. FLASK WEB APP
# ============================================

app = Flask(__name__)

# HTML Template (updated with the classes from your dataset)
def create_html_template(class_names):
    """Create HTML template with dynamic class names"""

    class_names_js = list(class_names)

    return f'''
<!DOCTYPE html>
<html>
<head>
    <title>Skin Disease Detection</title>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            min-height: 100vh;
            padding: 20px;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
            background: white;
            border-radius: 20px;
            padding: 30px;
            box-shadow: 0 20px 60px rgba(0,0,0,0.3);
        }}
        h1 {{
            text-align: center;
            color: #333;
            margin-bottom: 10px;
        }}
        .subtitle {{
            text-align: center;
            color: #666;
            margin-bottom: 30px;
        }}
        .camera-section {{
            display: flex;
            flex-direction: column;
            align-items: center;
            margin-bottom: 30px;
        }}
        .video-container {{
            position: relative;
            width: 100%;
            max-width: 640px;
            margin: 0 auto;
            border-radius: 15px;
            overflow: hidden;
            box-shadow: 0 10px 30px rgba(0,0,0,0.2);
        }}
        #video {{
            width: 100%;
            display: block;
            background: #f0f0f0;
        }}
        .camera-controls {{
            margin-top: 20px;
            display: flex;
            gap: 15px;
            flex-wrap: wrap;
            justify-content: center;
        }}
        button {{
            padding: 12px 30px;
            font-size: 16px;
            font-weight: 600;
            border: none;
            border-radius: 50px;
            cursor: pointer;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            transition: transform 0.2s;
        }}
        button:hover {{
            transform: translateY(-2px);
        }}
        button:disabled {{
            opacity: 0.5;
            cursor: not-allowed;
        }}
        .upload-section {{
            margin: 30px 0;
            text-align: center;
            padding: 30px;
            background: #f8f9fa;
            border-radius: 15px;
        }}
        .upload-label {{
            display: inline-block;
            padding: 15px 40px;
            background: linear-gradient(135deg, #43e97b 0%, #38f9d7 100%);
            color: white;
            border-radius: 50px;
            cursor: pointer;
            font-weight: 600;
            transition: transform 0.2s;
        }}
        .upload-label:hover {{
            transform: translateY(-2px);
        }}
        #imageUpload {{ display: none; }}
        .result-section {{
            margin-top: 30px;
            padding: 30px;
            background: linear-gradient(135deg, #f5f7fa 0%, #c3cfe2 100%);
            border-radius: 15px;
            display: none;
        }}
        .result-section.active {{ display: block; }}
        .prediction-card {{
            background: white;
            border-radius: 15px;
            padding: 25px;
            margin-bottom: 20px;
            text-align: center;
        }}
        .prediction-class {{
            font-size: 2em;
            color: #667eea;
            margin-bottom: 10px;
            font-weight: bold;
        }}
        .prediction-confidence {{
            font-size: 3em;
            font-weight: bold;
            color: #43e97b;
        }}
        .all-predictions {{
            margin-top: 20px;
        }}
        .prediction-bar {{
            margin: 10px 0;
            padding: 10px;
            background: #f8f9fa;
            border-radius: 10px;
        }}
        .bar {{
            height: 30px;
            background: linear-gradient(90deg, #667eea, #764ba2);
            border-radius: 15px;
            display: flex;
            align-items: center;
            padding: 0 10px;
            color: white;
            font-weight: bold;
            transition: width 0.5s;
        }}
        .loader {{
            border: 4px solid #f3f3f3;
            border-top: 4px solid #667eea;
            border-radius: 50%;
            width: 40px;
            height: 40px;
            animation: spin 1s linear infinite;
            margin: 20px auto;
            display: none;
        }}
        @keyframes spin {{
            0% {{ transform: rotate(0deg); }}
            100% {{ transform: rotate(360deg); }}
        }}
        .info {{
            text-align: center;
            margin-top: 20px;
            padding: 15px;
            background: #e3f2fd;
            border-radius: 10px;
        }}
        .footer {{
            text-align: center;
            margin-top: 30px;
            color: #666;
            font-size: 0.9em;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>🩺 Skin Disease Detection System</h1>
        <div class="subtitle">AI-Powered Dermatology Assistant</div>

        <div class="camera-section">
            <div class="video-container">
                <video id="video" autoplay playsinline></video>
            </div>
            <div class="camera-controls">
                <button id="startCamera" onclick="startCamera()">📷 Start Camera</button>
                <button id="captureBtn" onclick="captureImage()" disabled>📸 Capture & Analyze</button>
                <button id="switchCamera" onclick="switchCamera()" disabled>🔄 Switch Camera</button>
            </div>
        </div>

        <div class="upload-section">
            <label for="imageUpload" class="upload-label">📁 Upload Image for Analysis</label>
            <input type="file" id="imageUpload" accept="image/*" onchange="handleImageUpload(this)">
        </div>

        <div class="loader" id="loader"></div>

        <div class="result-section" id="results">
            <h2>📊 Analysis Results</h2>
            <img id="capturedImage" style="max-width: 100%; border-radius: 10px; margin-bottom: 20px;">
            <div id="predictionResult"></div>
        </div>

        <div class="info">
            💡 <strong>Note:</strong> This model detects: {', '.join(class_names_js)}
        </div>

        <div class="footer">⚕️ For educational and research purposes only. Always consult a dermatologist for medical advice.</div>
    </div>

    <script>
        let video = document.getElementById('video');
        let canvas = document.createElement('canvas');
        let context = canvas.getContext('2d');
        let currentStream = null;
        let usingFrontCamera = true;

        async function startCamera() {{
            try {{
                if (currentStream) {{
                    currentStream.getTracks().forEach(track => track.stop());
                }}
                const constraints = {{
                    video: {{ facingMode: usingFrontCamera ? 'user' : 'environment' }}
                }};
                currentStream = await navigator.mediaDevices.getUserMedia(constraints);
                video.srcObject = currentStream;
                document.getElementById('captureBtn').disabled = false;
                document.getElementById('switchCamera').disabled = false;
            }} catch (err) {{
                alert('Camera error: ' + err.message);
            }}
        }}

        function switchCamera() {{
            usingFrontCamera = !usingFrontCamera;
            startCamera();
        }}

        async function captureImage() {{
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            context.drawImage(video, 0, 0);
            const imageData = canvas.toDataURL('image/jpeg');
            await analyzeImage(imageData);
        }}

        async function handleImageUpload(input) {{
            if (input.files && input.files[0]) {{
                const reader = new FileReader();
                reader.onload = async function(e) {{
                    await analyzeImage(e.target.result);
                }};
                reader.readAsDataURL(input.files[0]);
            }}
        }}

        async function analyzeImage(imageData) {{
            document.getElementById('loader').style.display = 'block';
            document.getElementById('results').classList.remove('active');

            try {{
                const response = await fetch('/predict', {{
                    method: 'POST',
                    headers: {{ 'Content-Type': 'application/json' }},
                    body: JSON.stringify({{ image: imageData }})
                }});

                const data = await response.json();

                document.getElementById('loader').style.display = 'none';
                document.getElementById('results').classList.add('active');
                document.getElementById('capturedImage').src = imageData;

                // Display top prediction and all predictions
                let html = `
                    <div class="prediction-card">
                        <div class="prediction-class">🎯 Top Prediction: ${{data.top_prediction.class_name}}</div>
                        <div class="prediction-confidence">Confidence: ${{data.top_prediction.confidence}}%</div>
                        <div style="margin-top: 10px; color: #666;">${{data.top_prediction.description}}</div>
                    </div>
                    <div class="all-predictions">
                        <h3>All Predictions:</h3>
                `;

                for (const pred of data.all_predictions) {{
                    html += `
                        <div class="prediction-bar">
                            <div style="margin-bottom: 5px;">${{pred.class_name}}</div>
                            <div class="bar" style="width: ${{pred.confidence}}%;">
                                ${{pred.confidence}}%
                            </div>
                        </div>
                    `;
                }}

                html += `</div>`;
                document.getElementById('predictionResult').innerHTML = html;

            }} catch (error) {{
                document.getElementById('loader').style.display = 'none';
                alert('Error analyzing image: ' + error);
            }}
        }}

        // Auto-start camera when page loads
        window.onload = function() {{
            startCamera();
        }};
    </script>
</body>
</html>
    '''

def preprocess_image(image_data, target_size=(224, 224)):
    """Preprocess image for model prediction"""
    if ',' in image_data:
        image_data = image_data.split(',')[1]

    img_bytes = base64.b64decode(image_data)
    img = Image.open(BytesIO(img_bytes))

    if img.mode != 'RGB':
        img = img.convert('RGB')

    img = img.resize(target_size)
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    return img_array

# ============================================
# 8. MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    print("\n" + "=" * 50)
    print("🚀 Skin Disease Detection System")
    print("=" * 50)

    # Step 1: Mount Google Drive or download dataset
    print("\n📁 Accessing dataset...")

    # Try to mount Google Drive first (for Colab users)
    try:
        from google.colab import drive
        IN_COLAB = True
        drive.mount('/content/drive')
        dataset_path = "/content/drive/MyDrive/WYIE Picture"

        if not os.path.exists(dataset_path):
            print(f"⚠️ Dataset not found at {dataset_path}")
            print("Please ensure your 'WYIE Picture' folder is in the root of your Google Drive")
            dataset_path = "skin_dataset"  # Fallback
    except:
        IN_COLAB = False
        dataset_path = "skin_dataset"
        print("Not in Colab environment. Using local dataset path.")

    # Step 2: Prepare dataset
    train_gen, val_gen, test_gen = prepare_dataset(dataset_path)

    if train_gen is None:
        print("\n❌ Could not load dataset. Using demo mode with sample data.")
        print("\nTo fix this:")
        print("1. Upload your 'WYIE Picture' folder to Google Drive")
        print("2. Make sure it contains subfolders for each disease class")
        print("3. Each subfolder should contain images of that condition")
        print("\nOr create a structured dataset with train/val/test splits")

        # Create a simple demo model
        model = create_demo_model()
        class_names_list = ['Biduran', 'Bisul', 'Cacar Air', 'Dermatitis', 'Jerawat']
        class_names_dict = {i: (name, name) for i, name in enumerate(class_names_list)}

    else:
        # Step 3: Build and train model
        num_classes = train_gen.num_classes
        class_names_list = list(train_gen.class_indices.keys())

        print(f"\n📊 Building model for {num_classes} classes...")
        model = build_model(num_classes)
        model.summary()

        # Step 4: Train model
        model, history = train_model(model, train_gen, val_gen, epochs=20)

        # Step 5: Evaluate model
        print("\n" + "=" * 50)
        print("📊 Model Evaluation")
        print("=" * 50)

        test_loss, test_accuracy, test_auc = model.evaluate(test_gen)
        print(f"Test Accuracy: {test_accuracy:.4f}")
        print(f"Test AUC: {test_auc:.4f}")

        # Save model
        model.save('skin_disease_model.h5')
        print("\n✅ Model saved as 'skin_disease_model.h5'")

        # Create class names dictionary
        class_names_dict = {i: (name, name) for i, name in enumerate(class_names_list)}

    # Step 6: Start Flask app with ngrok
    print("\n" + "=" * 50)
    print("🌐 Starting Web Application")
    print("=" * 50)

    # Find available port
    PORT = find_free_port(start_port=5000)
    if PORT is None:
        print("❌ No available ports found")
        PORT = 5000

    print(f"✅ Using port: {PORT}")

    # Setup ngrok
    NGROK_AUTH_TOKEN = "2xLPMjjp35srucOx3q23Zo2MTkq_4ZoT4NAXBZDShAht5i7bk"

    try:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        print("✅ Ngrok auth token set successfully!")

        try:
            ngrok.kill()
            time.sleep(1)
        except:
            pass

        public_url = ngrok.connect(PORT)
        print(f"\n📱 PUBLIC URL (share this): {public_url}")
        print("⚠️  This URL will be active for 2-8 hours")

    except Exception as e:
        print(f"\n⚠️ Ngrok error: {e}")
        print("Running on localhost only")

    # Create Flask routes
    @app.route('/')
    def home():
        html_template = create_html_template(class_names_list)
        return render_template_string(html_template)

    @app.route('/predict', methods=['POST'])
    def predict():
        try:
            data = request.get_json()
            image_data = data['image']
            processed_img = preprocess_image(image_data)
            predictions = model.predict(processed_img, verbose=0)[0]

            # Get all predictions
            all_preds = []
            for i, prob in enumerate(predictions):
                confidence = float(prob * 100)
                all_preds.append({
                    'class_id': i,
                    'class_name': class_names_dict[i][0],
                    'description': class_names_dict[i][1],
                    'confidence': round(confidence, 2)
                })

            # Sort by confidence
            all_preds.sort(key=lambda x: x['confidence'], reverse=True)

            top_pred = all_preds[0]

            response = {
                'top_prediction': top_pred,
                'all_predictions': all_preds
            }
            return jsonify(response)

        except Exception as e:
            return jsonify({'error': str(e)}), 500

    # Start Flask server
    def run_app():
        app.run(host='0.0.0.0', port=PORT, debug=False, threaded=True)

    thread = threading.Thread(target=run_app)
    thread.daemon = True
    thread.start()

    print(f"\n✅ Server is running!")
    print(f"📍 Local URL: http://localhost:{PORT}")
    print("\n" + "=" * 50)
    print("Press Ctrl+C to stop the server")
    print("=" * 50)

    # Keep alive
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n\n🛑 Shutting down...")
        try:
            ngrok.kill()
        except:
            pass

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.2 MB/s eta 0:00:00
All libraries imported successfully!

🚀 Skin Disease Detection System

📁 Accessing dataset...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Creating train/val split from main directory...
Found 55 images belonging to 5 classes.
Found 12 images belonging to 5 classes.
Found 14 images belonging to 5 classes.

✅ Dataset loaded successfully!
Classes found: ['Biduran', 'Bisul', 'Cacar Air', 'Dermatitis ', 'Jerawat']
Training samples: 55
Validation samples: 12
Test samples: 14

📊 Building model for 5 classes...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,619,461 (9.99 MB)

 Trainable params: 361,477 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


Starting Training...
Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 13s 5s/step - accuracy: 0.2727 - auc: 0.5364 - loss: 2.0472 - val_accuracy: 0.3333 - val_auc: 0.7561 - val_loss: 1.3986 - learning_rate: 0.0010
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.2909 - auc: 0.6214 - loss: 1.6815 - val_accuracy: 0.5000 - val_auc: 0.8003 - val_loss: 1.3361 - learning_rate: 0.0010
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.3455 - auc: 0.7172 - loss: 1.4348 - val_accuracy: 0.5000 - val_auc: 0.7917 - val_loss: 1.3390 - learning_rate: 0.0010
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.3818 - auc: 0.7279 - loss: 1.4299 - val_accuracy: 0.5000 - val_auc: 0.7891 - val_loss: 1.3593 - learning_rate: 0.0010
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 3s/step - accuracy: 0.5273 - auc: 0.7964 - loss: 1.2259 - val_accuracy: 0.5833 - val_auc: 0.8212 - val_loss: 1.3186 - learning_rate: 0.0010
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.5455 - auc: 0.8013 - 

Test Accuracy: 0.5714
Test AUC: 0.8342

✅ Model saved as 'skin_disease_model.h5'

🌐 Starting Web Application
✅ Using port: 5000
✅ Ngrok auth token set successfully!

📱 PUBLIC URL (share this): NgrokTunnel: "https://a929-136-107-122-117.ngrok-free.app" -> "http://localhost:5000"
⚠️  This URL will be active for 2-8 hours

✅ Server is running!
📍 Local URL: http://localhost:5000

Press Ctrl+C to stop the server
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [27/Apr/2026 08:29:37] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Apr/2026 08:30:18] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Apr/2026 08:31:42] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Apr/2026 08:31:45] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Apr/2026 08:32:47] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Apr/2026 08:34:02] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Apr/2026 08:34:20] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [27/Apr/2026 08:34:56] "POST /predict HTTP/1.1" 200 -
